# OpenCourses.AI Colab version

This notebook belongs to **Fundamentos de Transformers** and was published as a public Colab-ready artifact.

Colab is an external free execution option with variable resources. Run the setup cell before executing the rest of the notebook.


In [ ]:
# OpenCourses.AI Colab setup
import os, pathlib, subprocess, sys
REPO_URL = "https://github.com/opencourses-ai-colab/opencourse-fundamentos-de-transformers-d9d23ccb.git"
BRANCH = "main"
TARGET = pathlib.Path("/content/opencourses/opencourse-fundamentos-de-transformers-d9d23ccb")
TARGET.parent.mkdir(parents=True, exist_ok=True)
if not TARGET.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(TARGET)], check=True)
os.environ['COURSE_ROOT'] = str(TARGET)
os.environ['ASSETS_DIR'] = str(TARGET / 'assets')
notebooks_dir = TARGET / 'notebooks'
if notebooks_dir.exists():
    os.chdir(notebooks_dir)
requirements = TARGET / 'requirements.txt'
if requirements.exists() and requirements.read_text(encoding='utf8').strip():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements)], check=True)
print(f'COURSE_ROOT={TARGET}')


<div style="border-bottom: 2px solid #1f2a44; padding-bottom: 14px; margin-bottom: 22px;">
  <div style="display: flex; align-items: center; justify-content: space-between; gap: 24px;">
    <div style="text-align: center; flex: 1; min-width: 260px;">
      <div style="font-size: 14px; letter-spacing: 0.04em; text-transform: uppercase; color: #5b6472;">Fundamentos de Transformers</div>
      <div style="font-size: 15px; font-weight: 700; color: #667085; margin-top: 6px;">Notebook 01</div>
      <div style="font-size: 26px; font-weight: 700; color: #1f2a44; margin-top: 2px;">El problema de modelar secuencias</div>
      <div style="font-size: 14px; color: #5b6472; margin-top: 8px;">Curso abierto</div>
    </div>
  </div>
</div>

<div style="display: flex; justify-content: space-between; gap: 16px; color: #3f4754; font-size: 14px; margin-bottom: 20px; flex-wrap: wrap;">
    <div><strong>Curso:</strong> Fundamentos de Transformers</div>
  <div><strong>Periodo:</strong> 2026-I</div>
</div>


## Presentación del notebook

<audio controls style="width: 100%; margin: 8px 0 18px 0;">
  <source src="../assets/audio/notebooks/01_el_problema_de_modelar_secuencias_intro.mp3" type="audio/mpeg">
  Su navegador no puede reproducir el audio embebido.
</audio>


## Pregunta directriz

> ¿Por qué un Transformer decoder aprende a predecir el siguiente token como una distribución local y no como una frase completa de una vez?

Este notebook instala el problema fundacional de la ruta: antes de hablar de atención, embeddings o capas, necesitamos entender qué tarea estadística resuelve un modelo autoregresivo.

## Objetivos

Al finalizar este notebook, el estudiante debería poder:

1. Explicar la diferencia entre una secuencia completa, una posición `t`, el contexto permitido y el token objetivo.
2. Leer la factorización autoregresiva como una cadena de decisiones locales.
3. Identificar en una tabla qué tokens son visibles, cuál es el objetivo y cuáles están prohibidos para una posición dada.
4. Distinguir logits, probabilidades y token observado dentro de una predicción del siguiente token.
5. Interpretar por qué esta tarea estadística no equivale por sí sola a comprensión lingüística general.

## Marco conceptual

Un Transformer decoder se entiende mejor si partimos del problema que resuelve: estimar una distribución sobre el siguiente token usando solo el pasado observable.

La arquitectura vendrá después. Primero aparece una restricción: para predecir la posición actual, el modelo no debe mirar el futuro. Esa restricción convierte el lenguaje en una secuencia de problemas locales: en cada posición, usar el contexto anterior para asignar probabilidades a los tokens posibles del vocabulario.

## Idea visual del problema

<div style="text-align: center; margin: 12px 0 18px 0;">
  <img src="../assets/figures/01_el_problema_de_modelar_secuencias.png" alt="Predicción autoregresiva como frontera móvil" style="width: min(980px, 98%); height: auto; border: 1px solid #d0d7de; border-radius: 6px;">
</div>

La figura resume la idea central: en cada posición hay una frontera móvil. A la izquierda queda el contexto permitido; en el centro, el token observado que sirve como objetivo; a la derecha, el futuro que no puede entrar en la predicción actual.

## Formulación matemática

Sea una secuencia discreta:

$$
x_{1:T}=(x_1,\ldots,x_T), \qquad x_t\in\mathcal{V}.
$$

Aquí $\mathcal{V}$ es el vocabulario, $x_t$ es el token observado en la posición $t$, y $x_{<t}$ representa todos los tokens anteriores a $t$.

Esta ecuación se lee así: una secuencia de longitud $T$ contiene un token discreto por posición, y cada token pertenece a un vocabulario finito.

Un modelo autoregresivo factoriza la probabilidad de la secuencia completa como:

$$
p_\theta(x_{1:T})=\prod_{t=1}^{T}p_\theta(x_t\mid x_{<t}).
$$

Esta ecuación se lee así: la probabilidad de la secuencia completa se descompone como el producto de muchas decisiones locales; en cada posición, el modelo estima la probabilidad del token actual usando solo los tokens anteriores.

Computacionalmente, cada factor $p_\theta(x_t\mid x_{<t})$ se obtiene en dos pasos: primero el modelo produce logits, que son puntuaciones no normalizadas sobre el vocabulario; luego una función softmax convierte esos logits en una distribución de probabilidad. El token observado es el objetivo de entrenamiento, no la única opción que el modelo consideró posible.

## Traducción tensorial

Una secuencia codificada aparece como un tensor de forma `(T,)`: una posición, un índice entero. Un batch agrupa varias secuencias y aparece como `(B,T)`, donde `B` es el número de ejemplos y `T` la longitud de contexto.

La restricción autoregresiva no está en el tipo de dato; está en qué posiciones se permiten como contexto. Más adelante, esa restricción se implementará con una máscara causal.

## Preparación del entorno

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from fundamentos_transformers.visualizacion import configurar_matplotlib

configurar_matplotlib()


## Experimento

Construiremos una frase corta y exploraremos distintas posiciones objetivo con un control interactivo. El deslizador cambia `t`: al moverlo se actualizan el contexto visible, el token que debe predecirse, el futuro no visible y una distribución simulada sobre el vocabulario.

La interacción busca que la causalidad autoregresiva deje de verse como una definición abstracta y pase a verse como una frontera móvil dentro de la secuencia.

La distribución de probabilidades del explorador es simulada. Sirve para leer logits, softmax y objetivo observado; no representa todavía un modelo entrenado.


In [2]:
from IPython.display import display
import importlib

import fundamentos_transformers.componentes_interactivos as componentes_interactivos

componentes_interactivos = importlib.reload(componentes_interactivos)

explorador = componentes_interactivos.crear_explorador_autoregresivo(
    texto="transformers modelan contexto",
    posiciones_referencia=[1, 2, 4, 8, 12, 16, 20, 24],
    posicion_inicial=8,
)
display(explorador)


## Interpretación

El explorador interactivo muestra la unidad real de entrenamiento de un modelo autoregresivo: no se le pide producir toda la frase de una sola vez, sino resolver muchas veces el mismo problema local. Al mover el deslizador, cambia la posición `t`; en cada caso, el modelo recibe únicamente los tokens anteriores y debe asignar una probabilidad a cada token posible del vocabulario.

La frontera vertical marca la restricción causal. A la izquierda está la información disponible; en rojo está el token observado que se usa como objetivo; a la derecha queda el futuro que el modelo no puede consultar. Esta separación es lo que impide que el modelo resuelva la tarea copiando información posterior.

Los logits del ejemplo deben leerse como puntuaciones comparables, no como probabilidades. Softmax convierte esas puntuaciones en una distribución sobre el vocabulario: varios tokens pueden tener probabilidad positiva, aunque solo uno sea el token observado en el texto. Como la distribución del explorador es simulada, el token observado no siempre aparece como el más probable; esto representa una predicción imperfecta o un modelo que todavía no ha aprendido bien ese contexto.

Durante el entrenamiento no se escoge el siguiente token: el texto ya trae el objetivo observado y la pérdida penaliza al modelo si le asigna poca probabilidad. Durante la generación sí aparece una regla de decisión: se puede escoger siempre el token más probable, o se puede muestrear entre alternativas probables usando estrategias como temperatura, top-k o top-p. Por eso conviene separar dos ideas: modelar una distribución y elegir una salida.

## Verificación de aprendizaje

Elija una posición `t` distinta en la frase del experimento y escriba:

1. el contexto permitido;
2. el token objetivo;
3. al menos tres tokens futuros que no deberían ser visibles;
4. una explicación breve de por qué esta separación implementa una restricción autoregresiva.

La respuesta debe mencionar explícitamente la diferencia entre logits, distribución de probabilidad y token observado.

## Síntesis

El primer problema del curso puede resumirse así:

$$
\text{secuencia} \rightarrow \text{contexto visible} \rightarrow \text{token objetivo} \rightarrow \text{distribución sobre vocabulario}.
$$

Esta cadena se lee así: en cada posición, el modelo recibe solo el pasado permitido y debe asignar puntajes a los posibles siguientes tokens. La restricción causal no es un detalle de implementación; define qué información puede usarse durante entrenamiento y generación.


## Preguntas de discusión

1. ¿Qué información falta si el contexto máximo es demasiado corto?
2. ¿Por qué predecir el siguiente token no equivale a comprender todo el lenguaje?
3. ¿Qué cambia entre entrenar con tokens observados y generar tokens nuevos paso a paso?

## Continuidad

El siguiente notebook convierte texto en tokens, índices y batches autoregresivos. Allí la frontera contexto-objetivo se materializa como dos tensores alineados: `X` y `Y`.